# 05 — DLT Expectations: The Lakehouse Answer to Engine-Level Constraints

In the RDBMS world, you had a layered defense:
- **CHECK constraints** caught basic data quality issues (nulls, format)
- **PRIMARY KEY / UNIQUE** prevented duplicates at the storage level
- **FOREIGN KEY** enforced referential integrity
- **Stored procedures & triggers** added custom business logic

On the lakehouse, **DLT (Delta Live Tables) expectations** replace all of that in a single,
declarative framework. You define your rules once in the pipeline, and DLT enforces them
on every run — no matter who or what upstream is writing data.

### Three enforcement modes (think: WARNING → FILTER → HARD STOP)

| DLT Mode | RDBMS Equivalent | Behavior |
|----------|------------------|----------|
| `expect` | A CHECK constraint that logs but doesn't reject | **Warn**: record the violation as a metric, keep the row |
| `expect_or_drop` | A BEFORE INSERT trigger that silently filters | **Drop**: remove violating rows, pipeline continues |
| `expect_or_fail` | A PRIMARY KEY / UNIQUE constraint that rejects the write | **Fail**: halt the entire pipeline, no bad data gets through |

### Where to enforce what (Bronze → Silver → Gold)

| Layer | Enforcement Level | Why |
|-------|-------------------|-----|
| **Bronze** | None — accept everything | Raw landing zone. You *want* duplicates here so you can debug what upstream sent you. Deleting them at ingest loses forensic visibility. |
| **Silver** | `expect_or_drop` — filter bad rows | Clean and conform. Drop nulls, malformed emails, obviously invalid records. Deduplicate with window functions. This is your staging area. |
| **Gold** | `expect_or_fail` — hard stop on violations | Business-ready data. If a duplicate PK makes it to Gold, something is fundamentally broken upstream and the pipeline should **stop**, not silently corrupt reporting. This is your PK/UNIQUE equivalent. |

This notebook:
1. Uploads `05_dlt_pipeline_code.py` to the workspace
2. Creates a DLT pipeline via the SDK
3. Runs it with clean data → pipeline **succeeds**
4. Runs it with duplicate data → pipeline **fails** on `expect_or_fail` (just like a PK violation would)
5. Cleans up

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import (
    PipelineLibrary,
    NotebookLibrary,
    PipelineCluster,
)
from dotenv import load_dotenv
import os
import time

load_dotenv()

w = WorkspaceClient()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "pk_unique_demo")

WORKSPACE_PATH = f"/Users/{w.current_user.me().user_name}/pk_unique_demo"
PIPELINE_NAME = "pk_unique_demo_dlt"

print(f"Workspace path: {WORKSPACE_PATH}")
print(f"Pipeline name: {PIPELINE_NAME}")

## Step 1: Upload pipeline code to workspace

In [ ]:
import base64

# Read the local pipeline code
with open("05_dlt_pipeline_code.py", "r") as f:
    pipeline_code = f.read()

# Ensure the workspace directory exists
w.workspace.mkdirs(WORKSPACE_PATH)

# Upload the pipeline code
remote_path = f"{WORKSPACE_PATH}/05_dlt_pipeline_code.py"
w.workspace.upload(
    path=remote_path,
    content=pipeline_code.encode("utf-8"),
    overwrite=True,
)

print(f"Uploaded pipeline code to: {remote_path}")

## Step 2: Create the DLT pipeline

In [ ]:
pipeline = w.pipelines.create(
    name=PIPELINE_NAME,
    serverless=True,
    channel="CURRENT",
    catalog=UC_CATALOG,
    target=f"{UC_SCHEMA}_dlt",
    libraries=[
        PipelineLibrary(
            notebook=NotebookLibrary(path=remote_path)
        )
    ],
    configuration={
        "uc_catalog": UC_CATALOG,
        "uc_schema": UC_SCHEMA,
    },
    continuous=False,
)

pipeline_id = pipeline.pipeline_id
print(f"Created pipeline: {pipeline_id}")

## Step 3: Prepare clean data and run the pipeline

First run: we'll feed in only the 10 genuinely new customers (no duplicates).
Think of this as a normal day — data arrives, pipeline validates it, Gold layer is clean.

This is equivalent to your Oracle stored procedure accepting a batch of valid INSERTs.

In [ ]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.serverless().getOrCreate()

# Save the full batch for later, then overwrite with clean-only data
spark.sql(f"""
    CREATE OR REPLACE TABLE {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full AS
    SELECT * FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch
""")

# Keep only genuinely new customers (customer_id > 200, unique emails)
spark.sql(f"""
    CREATE OR REPLACE TABLE {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch AS
    SELECT * FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full
    WHERE customer_id > 200
    AND email NOT IN (SELECT email FROM {UC_CATALOG}.{UC_SCHEMA}.customers)
""")

clean_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch").collect()[0].cnt
print(f"Clean batch: {clean_count} rows (no duplicates)")

In [ ]:
def run_pipeline_and_wait(pipeline_id, description=""):
    """Trigger a pipeline update and poll until completion."""
    print(f"Starting pipeline update{f' ({description})' if description else ''}...")
    update = w.pipelines.start_update(pipeline_id=pipeline_id, full_refresh=True)
    update_id = update.update_id
    print(f"Update ID: {update_id}")

    while True:
        status = w.pipelines.get(pipeline_id=pipeline_id)
        state = status.latest_updates[0].state.value if status.latest_updates else "UNKNOWN"
        print(f"  State: {state}")
        if state in ("COMPLETED", "FAILED", "CANCELED"):
            break
        time.sleep(15)

    return state


# Run with clean data
state = run_pipeline_and_wait(pipeline_id, "clean data — should succeed")
print(f"\nResult: {state}")

## Step 4: Swap in dirty data and run again

Now we restore the full batch — including 10 rows with duplicate `customer_id` and 10 with
duplicate `email`. This simulates a bad upstream feed or a bulk load gone wrong.

In Oracle, you'd get `ORA-00001: unique constraint violated` and the INSERT would be rejected.
Here, the DLT pipeline will **FAIL** at the Gold layer on `expect_or_fail` — same outcome,
different mechanism. The pipeline halts, no bad data reaches your consumers.

In [ ]:
# Restore the full batch with duplicates
spark.sql(f"""
    CREATE OR REPLACE TABLE {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch AS
    SELECT * FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full
""")

dirty_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch").collect()[0].cnt
print(f"Dirty batch: {dirty_count} rows (includes duplicates)")

# Run again — this should fail
state = run_pipeline_and_wait(pipeline_id, "dirty data — should FAIL")
print(f"\nResult: {state}")

## Step 5: Inspect the pipeline event log

In [ ]:
# Get recent events to show the expectation failure
events = list(w.pipelines.list_pipeline_events(
    pipeline_id=pipeline_id,
    max_results=20,
))

print("Recent pipeline events:")
print("=" * 80)
for event in events:
    if event.event_type in ("flow_progress", "update_progress"):
        print(f"\n[{event.event_type}] {event.timestamp}")
        if event.details:
            print(f"  {event.message}")

## The Full Picture: RDBMS Constraints → DLT Expectations

| What you had in Oracle/SQL Server | What you use on Databricks |
|-----------------------------------|----------------------------|
| `CHECK (email IS NOT NULL)` | `@dlt.expect_or_drop("email_not_null", "email IS NOT NULL")` |
| `PRIMARY KEY (customer_id)` | `@dlt.expect_or_fail("unique_customer_id", "id_count = 1")` at Gold |
| `UNIQUE (email)` | `@dlt.expect_or_fail("unique_email", "email_count = 1")` at Gold |
| `FOREIGN KEY ... REFERENCES` | `@dlt.expect_or_fail("valid_fk", "parent_exists = true")` at Gold |
| BEFORE INSERT trigger | `@dlt.expect_or_drop` at Silver (filter before it reaches Gold) |
| AFTER INSERT trigger / audit | `@dlt.expect` at any layer (log metric, keep the row for review) |
| Stored procedure with IF/THEN | The pipeline definition itself — declarative, not procedural |

### Why this is actually better than the RDBMS approach

1. **Centralized** — rules are declared once in the pipeline, not scattered across procs/triggers/app code
2. **Observable** — DLT tracks expectation metrics over time (% rows passing, failing, dropped) in the pipeline UI
3. **Graduated** — you choose warn/drop/fail per rule, per layer. In Oracle it's all-or-nothing reject
4. **Auditable** — Bronze keeps the raw data including violations, so you can always go back and investigate
5. **No bypass** — if all writes go through the DLT pipeline, there's no "someone did a raw INSERT" problem

### The recommendation

- **Day 1 (now):** Use `MERGE` + post-write validation in your existing PySpark/SQL jobs (notebook 04)
- **Day 2 (migration):** Move ingestion pipelines to DLT with `expect_or_drop` at Silver, `expect_or_fail` at Gold
- **Always:** Declare PK/UNIQUE/FK constraints in your DDL for optimizer hints and BI tool compatibility, even though they're not enforced

## Cleanup

In [ ]:
# Delete the DLT pipeline
w.pipelines.delete(pipeline_id=pipeline_id)
print(f"Deleted pipeline: {pipeline_id}")

# Clean up the backup table
spark.sql(f"DROP TABLE IF EXISTS {UC_CATALOG}.{UC_SCHEMA}.new_customers_batch_full")

# Clean up the DLT target schema
spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{UC_SCHEMA}_dlt CASCADE")

# Remove uploaded workspace file
try:
    w.workspace.delete(path=WORKSPACE_PATH, recursive=True)
    print(f"Deleted workspace folder: {WORKSPACE_PATH}")
except Exception as e:
    print(f"Note: could not delete workspace folder: {e}")

print("\nCleanup complete!")